Traditional measures like IMD tell us gentrification happened; 

Our cascading flow index shows how it happened — through sequential displacement.

### Merge Data

In this script, we will merge 2021 census data of origin-destination and IMD data (2019).

Since census O-D data is at MSOA level, while IMD data is at LSOA level, we will use a lookup table to merge them together.

In [1]:
import pandas as pd

from pyprojroot import here

In [9]:
ROOT = here()

DATA_DIR = ROOT / "data"

imd_path = DATA_DIR / "imd.csv"
census_od_path = DATA_DIR / "ODMG01EW_MSOA.csv"
lookup_path = DATA_DIR / "NSPCL_NOV22_UK_LU.csv"

In [ ]:
# 1. Loading Datasets

imd_lsoa = pd.read_csv(imd_path)
census_od = pd.read_csv(census_od_path)
lookup = pd.read_csv(lookup_path, 
                           encoding='ISO-8859-1',
                           low_memory=False)

In [19]:
print(lookup['ladnm'].unique()[:20])

['Aberdeen City' 'Aberdeenshire' nan 'Angus' 'Moray' 'Highland'
 'St Albans' 'Welwyn Hatfield' 'Hertsmere' 'Dacorum'
 'Central Bedfordshire' 'North Hertfordshire' 'East Hertfordshire'
 'Birmingham' 'Solihull' 'Bromsgrove' 'Sandwell' 'Walsall'
 'North Warwickshire' 'Stratford-on-Avon']


In [ ]:
# 2. Filter areas of London

# The whole 33 London boroughs
london_boroughs = [
    'City of London', 'Barking and Dagenham', 'Barnet', 'Bexley', 'Brent', 
    'Bromley', 'Camden', 'Croydon', 'Ealing', 'Enfield', 'Greenwich', 
    'Hackney', 'Hammersmith and Fulham', 'Haringey', 'Harrow', 'Havering', 
    'Hillingdon', 'Hounslow', 'Islington', 'Kensington and Chelsea', 
    'Kingston upon Thames', 'Lambeth', 'Lewisham', 'Merton', 'Newham', 
    'Redbridge', 'Richmond upon Thames', 'Southwark', 'Sutton', 
    'Tower Hamlets', 'Waltham Forest', 'Wandsworth', 'Westminster'
]

# Filter London boroughs with specified names
london_lookup = lookup[lookup['ladnm'].isin(london_boroughs)]

In [29]:
imd_lsoa.columns = imd_lsoa.columns.str.strip()
print(lookup_table.columns.tolist())

['pcd7', 'pcd8', 'pcds', 'dointr', 'doterm', 'usertype', 'oseast1m', 'osnrth1m', 'oa11cd', 'oac11cd', 'oac11nm', 'wz11cd', 'wzc11cd', 'wzc11nm', 'lsoa11cd', 'lsoa11nm', 'msoa11cd', 'msoa11nm', 'soac11cd', 'soac11nm', 'ladcd', 'ladnm', 'ladnmw', 'laccd', 'lacnm']


In [ ]:
# 3. Aggregate IMD to MSOA-Level

# Merge IMD scores into the london_lookup
imd_msoa_map = pd.merge(
    imd_lsoa[['LSOA code (2011)', 'Index of Multiple Deprivation (IMD) Score']], 
    london_lookup[['lsoa11cd', 'msoa11cd', 'ladnm']], 
    left_on='LSOA code (2011)', 
    right_on='lsoa11cd'
)

# Calculate IMD average scores for each MSOA
msoa_wealth = imd_msoa_map.groupby(['msoa11cd', 'ladnm'])['Index of Multiple Deprivation (IMD) Score'].mean().reset_index()

In [ ]:
# 4. Startify wealth deciles

msoa_wealth['Wealth_Decile'] = pd.qcut(msoa_wealth['Index of Multiple Deprivation (IMD) Score'], 10, labels=False) + 1
msoa_wealth['Wealth_Decile'] = 11 - msoa_wealth['Wealth_Decile']

In [33]:
# Prepare for match with census-od data

wealth_dict = msoa_wealth.set_index('msoa11cd')['Wealth_Decile'].to_dict()

print(msoa_wealth[['msoa11cd', 'ladnm', 'Wealth_Decile']].head())

    msoa11cd                 ladnm  Wealth_Decile
0  E02000001        City of London              8
1  E02000002  Barking and Dagenham              1
2  E02000003  Barking and Dagenham              4
3  E02000004  Barking and Dagenham              5
4  E02000005  Barking and Dagenham              3


In [34]:
# 5. Clean the census O-D data

od_clean = census_od[census_od['Migrant MSOA one year ago code'] != '-8'].copy()

In [36]:
# 6. Match deciles with O-D data

# Match w/ origins 
od_clean['Origin_Decile'] = od_clean['Migrant MSOA one year ago code'].map(wealth_dict)

# Match w/ destinations
od_clean['Dest_Decile'] = od_clean['Middle layer Super Output Areas code'].map(wealth_dict)

In [37]:
# 7. Filter flows only within London

london_flow = od_clean.dropna(subset=['Origin_Decile', 'Dest_Decile']).copy()

In [ ]:
# 8. Calculate flow features for each MSOA

msoa_analysis = msoa_wealth[['msoa11cd', 'ladnm', 'Wealth_Decile']].copy()

# --- A. Inflow ---
# For each MSOA, calculate how many ppl are moved from wealthier MSOAs
